# RSNA Knee Abnormality Detection — Exploratory Data Analysis

Kompetisi: [rsna-knee-abnormality-detection](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection)

**Tugas:** memprediksi probabilitas 12 abnormalitas lutut per studi MRI:
`ACL, MCL, Medial Meniscus, Lateral Meniscus, Medial OA, Lateral OA, PF OA, Effusion, Synovitis, Baker's, Contusion, Fracture`.

**Struktur data (hasil investigasi awal via Kaggle API):**

| File | Isi |
|---|---|
| `train.csv` | `StudyInstanceUID`, `Report` (teks laporan radiologi, multi-bahasa), + 12 kolom label |
| `train_series.csv` | `StudyInstanceUID`, `SeriesInstanceUID`, `Fluid_Sensitive`, `Fat_Suppression`, `Anatomical_Plane` |
| `train_series/<StudyUID>/<SeriesUID>/*.dcm` | file DICOM per-slice |
| `test.csv` | `StudyInstanceUID` — **sampel kecil** (indikasi code-competition: test asli tersembunyi saat scoring) |
| `test_series.csv` | sama seperti `train_series.csv` tapi untuk studi test |
| `test_series/<StudyUID>/<SeriesUID>/*.dcm` | file DICOM test |
| `sample_submission.csv` | `StudyInstanceUID` + 12 kolom label, semua diisi `0.5` (probabilitas) |

**Temuan penting dari eksplorasi awal (sampel parsial via API):**
- ⚠️ **~98–99% label pada `train.csv` kosong (NaN)** — hanya sebagian kecil studi yang punya label pathology terstruktur. Sisanya *hanya* punya `Report` teks bebas. ini mengindikasikan skema **weak/semi-supervised learning**: sebagian besar sinyal ada di teks laporan radiologi, bukan label terstruktur.
- Laporan bersifat **multi-bahasa** (setidaknya Spanish, Dutch/German, English terlihat pada sampel).
- Setiap studi MRI punya banyak *series* (potongan Axial/Sagittal/Coronal, dengan/tanpa fat-suppression, fluid-sensitive).
- `test.csv` versi publik cuma berisi beberapa baris — pola khas *code competition* Kaggle (test asli disembunyikan saat re-run scoring).

Notebook ini didesain untuk **dijalankan langsung di Kaggle** (kernel notebook) supaya punya akses penuh & cepat ke seluruh dataset (termasuk ribuan file DICOM), lalu hasil eksekusinya (grafik, output) ditarik kembali ke lokal.

## 1. Setup

In [ ]:
import os, sys, glob, random, warnings, textwrap
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 160)

RNG = np.random.default_rng(42)
random.seed(42)

# Optional deps used later — install quietly if missing (works when internet is enabled on the kernel)
def ensure(pkg, import_name=None):
    import importlib
    name = import_name or pkg
    try:
        importlib.import_module(name)
        return True
    except ImportError:
        try:
            os.system(f"{sys.executable} -m pip install -q {pkg}")
            importlib.import_module(name)
            return True
        except Exception as e:
            print(f"[warn] could not import/install {pkg}: {e}")
            return False

HAVE_PYDICOM = ensure("pydicom")
HAVE_LANGDETECT = ensure("langdetect")
if HAVE_PYDICOM:
    import pydicom
if HAVE_LANGDETECT:
    from langdetect import detect, DetectorFactory
    DetectorFactory.seed = 42

# Knee MRI DICOMs are frequently stored with a *compressed* transfer syntax
# (JPEG2000/JPEG-LS/RLE). pydicom needs one of these codec backends to decode
# `.pixel_array` in that case, otherwise pixel access raises/returns blank data.
for _pkg in ("pylibjpeg", "pylibjpeg-libjpeg", "pylibjpeg-openjpeg", "python-gdcm"):
    ensure(_pkg, _pkg.replace("-", "_") if _pkg != "python-gdcm" else "gdcm")


In [ ]:
# Plot style — consistent, readable in Kaggle's light theme
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#444444",
    "axes.grid": True,
    "grid.color": "#e6e6e6",
    "grid.linewidth": 0.8,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
})
ACCENT = "#3B7DD8"
ACCENT2 = "#D8763B"
PALETTE = ["#3B7DD8", "#D8763B", "#4CA36B", "#B24C63", "#7A5CC0", "#C9A227", "#4CA3A3", "#8C8C8C"]

OUT_DIR = Path("eda_outputs")
OUT_DIR.mkdir(exist_ok=True)

def savefig(fig, name, **kw):
    fig.tight_layout()
    fig.savefig(OUT_DIR / name, dpi=140, bbox_inches="tight", **kw)


## 2. Lokasi data & load tabel metadata

Mendukung dua mode path: berjalan di Kaggle (`/kaggle/input/...`) atau lokal (folder `data/` di samping notebook). Path di-*auto-discover* (bukan hardcode slug) supaya tahan terhadap variasi nama folder mount.

In [ ]:
# Diagnostik: apa saja yang benar-benar ter-mount di /kaggle/input?
kaggle_input = Path("/kaggle/input")
if kaggle_input.exists():
    print("/kaggle/input isi:", sorted(p.name for p in kaggle_input.iterdir()))
else:
    print("/kaggle/input tidak ada (kemungkinan berjalan bukan di Kaggle).")

In [ ]:
CANDIDATE_ROOTS = [
    Path("/kaggle/input/rsna-knee-abnormality-detection"),
    Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
    Path("../data"),
    Path("data"),
]
# auto-discover: cari train.csv/test.csv di /kaggle/input, dibatasi 2 level ke bawah saja.
# PENTING: sengaja TIDAK pakai rglob() rekursif tanpa batas — dataset ini punya puluhan ribu
# folder DICOM (train_series/<Study>/<Series>/*.dcm), jadi rglob bisa menjelajah semuanya dan
# menggantung lama di FUSE mount Kaggle. glob dangkal (depth 1-2) sudah cukup untuk menemukan
# pola /kaggle/input/<slug>/ maupun /kaggle/input/competitions/<slug>/.
if kaggle_input.exists():
    for pattern in ("*/train.csv", "*/test.csv", "*/*/train.csv", "*/*/test.csv"):
        for hit in kaggle_input.glob(pattern):
            CANDIDATE_ROOTS.insert(0, hit.parent)

DATA_DIR = next((p for p in CANDIDATE_ROOTS if p.exists() and ((p / "train.csv").exists() or (p / "test.csv").exists())), None)
if DATA_DIR is None:
    DATA_DIR = next((p for p in CANDIDATE_ROOTS if p.exists()), CANDIDATE_ROOTS[0])

print("Using DATA_DIR:", DATA_DIR, "| exists:", DATA_DIR.exists())
if DATA_DIR.exists():
    print(sorted(p.name for p in DATA_DIR.iterdir())[:20])


In [ ]:
def read_csv_safe(name):
    p = DATA_DIR / name
    if not p.exists():
        print(f"[missing] {name}")
        return None
    df = pd.read_csv(p)
    print(f"{name}: shape={df.shape}")
    return df

train = read_csv_safe("train.csv")
train_series = read_csv_safe("train_series.csv")
test = read_csv_safe("test.csv")
test_series = read_csv_safe("test_series.csv")
sample_sub = read_csv_safe("sample_submission.csv")

LABEL_COLS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
              "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
              "Contusion", "Fracture"]
LABEL_COLS = [c for c in LABEL_COLS if train is not None and c in train.columns]
print("Label columns found:", LABEL_COLS)

HAVE_TRAIN = train is not None and len(LABEL_COLS) > 0
if not HAVE_TRAIN:
    print("[!] train.csv / kolom label tidak ditemukan — bagian yang bergantung padanya akan di-skip otomatis di bawah.")


In [ ]:
# Sanity check: duplicate StudyInstanceUID (bisa muncul kalau file rusak / re-run parsial)
if train is not None:
    dup = train['StudyInstanceUID'].duplicated().sum()
    print('Duplicate StudyInstanceUID in train.csv:', dup)
    if dup:
        train = train.drop_duplicates(subset='StudyInstanceUID').reset_index(drop=True)
        print('-> dropped, new shape:', train.shape)


In [ ]:
train.head(8) if train is not None else None

In [ ]:
test.head() if test is not None else None

In [ ]:
sample_sub.head() if sample_sub is not None else None

In [ ]:
train_series.head() if train_series is not None else None

## 3. Overview ukuran dataset & struktur hierarki

Studi → Series (potongan MRI: Axial/Sagittal/Coronal, dengan variasi urutan pulsa) → Instance (slice DICOM individual).

In [ ]:
summary_rows = []
if train is not None:
    summary_rows.append(("train studies", train['StudyInstanceUID'].nunique()))
if test is not None:
    summary_rows.append(("test studies (public sample)", test['StudyInstanceUID'].nunique()))
if train_series is not None:
    summary_rows.append(("train series", len(train_series)))
    summary_rows.append(("avg series / train study", round(len(train_series) / train['StudyInstanceUID'].nunique(), 2) if train is not None else None))
if test_series is not None:
    summary_rows.append(("test series (public sample)", len(test_series)))

summary_df = pd.DataFrame(summary_rows, columns=["metric", "value"])
summary_df

> **Catatan tentang `test.csv`:** jika jumlah studi test publik sangat kecil (mis. hanya beberapa), ini adalah pola *code competition* khas Kaggle — file yang terlihat cuma contoh format; test set asli (jauh lebih besar) disembunyikan dan baru diberikan saat notebook submission di-*rerun* oleh Kaggle untuk scoring. Jangan asumsikan distribusi test dari sampel publik ini.

## 4. Label / target EDA

### 4.1 Missingness — proporsi label yang terisi vs kosong

In [ ]:
if HAVE_TRAIN:
    miss = train[LABEL_COLS].isna().mean().sort_values() * 100
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.barh(miss.index, 100 - miss.values, color=ACCENT)
    ax.set_xlabel("% studi dengan label TERISI (non-null)")
    ax.set_title("Ketersediaan label per pathology — train.csv")
    ax.set_xlim(0, max(5, (100 - miss.values).max() * 1.3))
    for b, v in zip(bars, 100 - miss.values):
        ax.text(b.get_width() + 0.1, b.get_y() + b.get_height()/2, f"{v:.1f}%", va="center", fontsize=9)
    savefig(fig, "01_label_availability.png")
    plt.show()

    print(f"Rata-rata seluruh kolom label kosong: {miss.mean():.1f}%")
    n_any = (~train[LABEL_COLS].isna().all(axis=1)).sum()
    print(f"Studi dengan MINIMAL 1 label terisi: {n_any} / {len(train)} ({n_any/len(train)*100:.2f}%)")
else:
    print("[skip] train.csv tidak tersedia.")

### 4.2 Distribusi kelas pada subset studi yang berlabel

In [ ]:
if HAVE_TRAIN:
    labeled = train.dropna(subset=LABEL_COLS, how="all")
    print("Jumlah studi berlabel (>=1 label terisi):", len(labeled))

    fig, axes = plt.subplots(3, 4, figsize=(16, 9))
    for ax, col in zip(axes.flat, LABEL_COLS):
        vc = labeled[col].value_counts(dropna=True).reindex([0, 1]).fillna(0)
        ax.bar(["Negatif (0)", "Positif (1)"], vc.values, color=[PALETTE[0], PALETTE[1]])
        ax.set_title(col, fontsize=11)
        n_lab = int(vc.sum())
        ax.set_xlabel(f"n={n_lab}", fontsize=9)
        for i, v in enumerate(vc.values):
            ax.text(i, v + 0.3, int(v), ha="center", fontsize=9)
    fig.suptitle("Class balance per label (hanya baris yang punya nilai untuk kolom tsb)", y=1.02, fontsize=13, fontweight="bold")
    savefig(fig, "02_label_class_balance.png")
    plt.show()
else:
    print("[skip] train.csv tidak tersedia.")

### 4.3 Ko-okurensi antar label (di subset berlabel)

In [ ]:
if HAVE_TRAIN:
    co = labeled[LABEL_COLS].fillna(0).astype(int)
    corr = co.corr()

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(LABEL_COLS))); ax.set_xticklabels(LABEL_COLS, rotation=90)
    ax.set_yticks(range(len(LABEL_COLS))); ax.set_yticklabels(LABEL_COLS)
    for i in range(len(LABEL_COLS)):
        for j in range(len(LABEL_COLS)):
            ax.text(j, i, f"{corr.values[i,j]:.1f}", ha="center", va="center", fontsize=7,
                    color="white" if abs(corr.values[i,j]) > 0.5 else "black")
    fig.colorbar(im, ax=ax, shrink=0.8, label="korelasi Pearson (0/1)")
    ax.set_title("Korelasi antar label pathology (subset berlabel)")
    savefig(fig, "03_label_correlation.png")
    plt.show()
else:
    print("[skip] train.csv tidak tersedia.")

In [ ]:
if HAVE_TRAIN:
    n_positive_per_study = co.sum(axis=1)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(n_positive_per_study, bins=range(0, len(LABEL_COLS) + 2), color=ACCENT, align="left", rwidth=0.85)
    ax.set_xlabel("Jumlah label positif per studi")
    ax.set_ylabel("Jumlah studi")
    ax.set_title("Berapa banyak abnormalitas ditemukan bersamaan per studi?")
    savefig(fig, "04_labels_per_study_hist.png")
    plt.show()
else:
    print("[skip] train.csv tidak tersedia.")

## 5. Radiology report — text EDA

Karena label terstruktur sangat jarang, sinyal utama untuk sebagian besar studi ada di teks `Report`.

In [ ]:
HAVE_REPORT = train is not None and 'Report' in train.columns
if HAVE_REPORT:
    has_report = train['Report'].notna()
    print(f"Studi dengan Report terisi: {has_report.sum()} / {len(train)} ({has_report.mean()*100:.1f}%)")

    train['report_len_chars'] = train['Report'].str.len()
    train['report_len_words'] = train['Report'].str.split().str.len()

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    axes[0].hist(train['report_len_chars'].dropna(), bins=40, color=ACCENT)
    axes[0].set_title("Panjang laporan (karakter)")
    axes[0].set_xlabel("jumlah karakter")
    axes[1].hist(train['report_len_words'].dropna(), bins=40, color=ACCENT2)
    axes[1].set_title("Panjang laporan (kata)")
    axes[1].set_xlabel("jumlah kata")
    savefig(fig, "05_report_length.png")
    plt.show()

    display(train['report_len_words'].describe())
else:
    print("[skip] kolom Report tidak tersedia.")

### 5.1 Bahasa laporan (heuristik `langdetect`, sampel)

Laporan radiologi di dataset ini teramati multi-bahasa (Spanish, Dutch/German, English, dll). Deteksi bahasa membantu memahami distribusi asal data & tantangan NLP multi-bahasa.

In [ ]:
def detect_lang(text):
    if not isinstance(text, str) or len(text.strip()) < 8:
        return None
    try:
        return detect(text)
    except Exception:
        return None

if HAVE_LANGDETECT and HAVE_REPORT:
    sample_n = min(1500, has_report.sum())
    sample_idx = train.loc[has_report].sample(sample_n, random_state=42).index
    langs = train.loc[sample_idx, 'Report'].apply(detect_lang)
    lang_counts = langs.value_counts(dropna=True)

    fig, ax = plt.subplots(figsize=(8, 4.5))
    top = lang_counts.head(12)
    ax.bar(top.index, top.values, color=PALETTE[: len(top)])
    ax.set_title(f"Distribusi bahasa laporan (sampel n={sample_n})")
    ax.set_ylabel("jumlah laporan")
    savefig(fig, "06_report_language.png")
    plt.show()
    print(lang_counts)
else:
    print("[skip] langdetect dan/atau kolom Report tidak tersedia.")

### 5.2 Kata/frasa yang sering muncul, dan kaitannya dengan label

Mengecek apakah kata kunci sederhana pada laporan berkorelasi dengan label terstruktur — indikasi awal kelayakan pendekatan *weak supervision* dari teks laporan.

In [ ]:
import re

KEYWORDS = {
    "ACL": ["acl", "anterior cruciate", "ligamento cruzado anterior", "cruciate anterior", "kruisband"],
    "MCL": ["mcl", "medial collateral", "ligamento colateral medial"],
    "Medial Meniscus": ["medial meniscus", "menisco medial", "menisco interno", "meniscus medialis"],
    "Lateral Meniscus": ["lateral meniscus", "menisco lateral", "menisco externo"],
    "Effusion": ["effusion", "derrame", "joint effusion", "erguss"],
    "Fracture": ["fracture", "fractura", "fraktur"],
    "Synovitis": ["synovitis", "sinovitis"],
    "Contusion": ["contusion", "contusión", "bone bruise", "bone marrow edema"],
}

if HAVE_TRAIN and HAVE_REPORT:
    rows = []
    report_lower = train['Report'].str.lower().fillna("")
    for label, kws in KEYWORDS.items():
        if label not in LABEL_COLS:
            continue
        pattern = re.compile("|".join(re.escape(k) for k in kws))
        sub = train.loc[train[label].notna()]
        if len(sub) == 0:
            continue
        kw_sub = report_lower.loc[sub.index].str.contains(pattern)
        rate_pos = sub.loc[kw_sub, label].mean() if kw_sub.sum() else np.nan
        rate_neg = sub.loc[~kw_sub, label].mean() if (~kw_sub).sum() else np.nan
        rows.append({
            "label": label,
            "n_labeled": len(sub),
            "n_with_keyword": int(kw_sub.sum()),
            "P(label=1 | keyword hadir)": rate_pos,
            "P(label=1 | keyword absen)": rate_neg,
        })

    keyword_check = pd.DataFrame(rows).round(3)
    display(keyword_check)
else:
    print("[skip] train.csv / kolom Report tidak tersedia.")

*(Jika `P(label=1 | keyword hadir)` jauh lebih tinggi dari `P(label=1 | keyword absen)`, itu sinyal bahwa keyword sederhana dari laporan cukup prediktif — mendukung strategi weak-labeling / distant supervision memakai teks laporan untuk memperbanyak data latih di luar 12 label yang jarang terisi.)*

## 6. Series-level metadata EDA (`train_series.csv` / `test_series.csv`)

In [ ]:
if train_series is not None:
    n_series_per_study = train_series.groupby('StudyInstanceUID').size()
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.hist(n_series_per_study, bins=range(1, n_series_per_study.max() + 2), color=ACCENT, align="left", rwidth=0.85)
    ax.set_xlabel("jumlah series per studi")
    ax.set_ylabel("jumlah studi")
    ax.set_title("Berapa banyak MRI series per studi?")
    savefig(fig, "07_series_per_study.png")
    plt.show()
    print(n_series_per_study.describe())

In [ ]:
if train_series is not None:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    plane_counts = train_series['Anatomical_Plane'].value_counts()
    axes[0].bar(plane_counts.index, plane_counts.values, color=PALETTE[:len(plane_counts)])
    axes[0].set_title("Anatomical_Plane")
    axes[0].tick_params(axis='x', rotation=20)

    fs_counts = train_series['Fluid_Sensitive'].value_counts().sort_index()
    axes[1].bar(fs_counts.index.astype(str), fs_counts.values, color=PALETTE[2:4])
    axes[1].set_title("Fluid_Sensitive")

    fat_counts = train_series['Fat_Suppression'].value_counts().sort_index()
    axes[2].bar(fat_counts.index.astype(str), fat_counts.values, color=PALETTE[4:6])
    axes[2].set_title("Fat_Suppression")

    fig.suptitle("Karakteristik sekuens MRI per series (train)", fontsize=13, fontweight="bold")
    savefig(fig, "08_series_sequence_flags.png")
    plt.show()

In [ ]:
if train_series is not None:
    cross = pd.crosstab(train_series['Anatomical_Plane'],
                         [train_series['Fluid_Sensitive'], train_series['Fat_Suppression']])
    cross.columns = [f"FS={a},FatSup={b}" for a, b in cross.columns]
    cross

### 6.1 Train vs Test — apakah distribusi series sequence sebanding?

(Ingat: `test_series.csv` publik hanya mencakup sampel kecil studi test — perbandingan ini indikatif, bukan definitif.)

In [ ]:
if train_series is not None and test_series is not None and len(test_series) > 0:
    comp = pd.DataFrame({
        "train_%": train_series['Anatomical_Plane'].value_counts(normalize=True) * 100,
        "test_%": test_series['Anatomical_Plane'].value_counts(normalize=True) * 100,
    }).fillna(0).round(1)

    fig, ax = plt.subplots(figsize=(7, 4.5))
    x = np.arange(len(comp))
    w = 0.35
    ax.bar(x - w/2, comp['train_%'], width=w, label="train", color=ACCENT)
    ax.bar(x + w/2, comp['test_%'], width=w, label="test (sampel publik)", color=ACCENT2)
    ax.set_xticks(x); ax.set_xticklabels(comp.index)
    ax.set_ylabel("% series")
    ax.set_title("Anatomical_Plane: train vs test (sampel publik)")
    ax.legend()
    savefig(fig, "09_train_vs_test_plane.png")
    plt.show()
    comp

## 7. DICOM image-level EDA

Membaca *sampel* file DICOM (bukan seluruh dataset — bisa puluhan ribu file) untuk memeriksa metadata pencitraan dan menampilkan contoh slice per bidang anatomis.

In [ ]:
def series_dir_candidates(root_name):
    d = DATA_DIR / root_name
    return d if d.exists() else None

TRAIN_SERIES_DIR = series_dir_candidates("train_series")
print("train_series dir:", TRAIN_SERIES_DIR)

def list_dcm_files(study_uid, series_uid, base=TRAIN_SERIES_DIR, limit=None):
    d = base / study_uid / series_uid
    if not d.exists():
        return []
    files = sorted(d.glob("*.dcm"))
    return files[:limit] if limit else files

In [ ]:
SAMPLE_N_SERIES = 8
if train_series is not None and HAVE_PYDICOM and TRAIN_SERIES_DIR is not None:
    sample_series = train_series.sample(min(SAMPLE_N_SERIES, len(train_series)), random_state=42)

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    meta_rows = []
    for ax, (_, row) in zip(axes.flat, sample_series.iterrows()):
        files = list_dcm_files(row['StudyInstanceUID'], row['SeriesInstanceUID'], limit=None)
        ax.axis("off")
        if not files:
            ax.set_title("file tidak ditemukan", fontsize=9)
            continue
        mid_file = files[len(files) // 2]  # ambil slice tengah, biasanya paling representatif
        ds = pydicom.dcmread(mid_file, force=True)
        title = f"{row['Anatomical_Plane']} | FS={row['Fluid_Sensitive']} FatSup={row['Fat_Suppression']}"
        try:
            img = ds.pixel_array.astype(float)
            img = (img - img.min()) / (np.ptp(img) + 1e-6)  # np.ptp(), bukan .ptp() method (dihapus di NumPy 2.0+)
            ax.imshow(img, cmap="gray")
        except Exception as e:
            print(f"[warn] gagal decode pixel_array untuk {mid_file.name}: {e}")
            title += f"\n[gagal render: {type(e).__name__}]"
        ax.set_title(title, fontsize=9)

        meta_rows.append({
            "StudyInstanceUID": row['StudyInstanceUID'][-12:],
            "n_instances": len(files),
            "Rows": getattr(ds, "Rows", None),
            "Columns": getattr(ds, "Columns", None),
            "PixelSpacing": getattr(ds, "PixelSpacing", None),
            "SliceThickness": getattr(ds, "SliceThickness", None),
            "Modality": getattr(ds, "Modality", None),
            "MagneticFieldStrength": getattr(ds, "MagneticFieldStrength", None),
            "Manufacturer": getattr(ds, "Manufacturer", None),
        })

    fig.suptitle(f"Contoh slice tengah dari {len(sample_series)} series (sampel acak)", fontsize=13, fontweight="bold")
    savefig(fig, "10_sample_slices_grid.png")
    plt.show()

    dicom_meta = pd.DataFrame(meta_rows)
    display(dicom_meta)
else:
    print("Lewati — pydicom / direktori train_series tidak tersedia di environment ini.")

In [ ]:
# Distribusi jumlah instance (slice) per series — disampel agar cepat, karena bisa ada ribuan series
if train_series is not None and TRAIN_SERIES_DIR is not None:
    SAMPLE_N = min(300, len(train_series))
    sampled = train_series.sample(SAMPLE_N, random_state=7)
    counts = []
    for _, row in sampled.iterrows():
        d = TRAIN_SERIES_DIR / row['StudyInstanceUID'] / row['SeriesInstanceUID']
        if d.exists():
            counts.append(len(list(d.glob("*.dcm"))))
    counts = pd.Series(counts)
    if len(counts):
        fig, ax = plt.subplots(figsize=(7, 4.5))
        ax.hist(counts, bins=30, color=ACCENT)
        ax.set_xlabel("jumlah instance (slice) per series")
        ax.set_ylabel("jumlah series (sampel)")
        ax.set_title(f"Distribusi jumlah slice per series (n={len(counts)} sampel)")
        savefig(fig, "11_instances_per_series_hist.png")
        plt.show()
        print(counts.describe())


In [ ]:
# Histogram intensitas pixel dari beberapa contoh slice (untuk cek rentang nilai / perlunya windowing)
if train_series is not None and HAVE_PYDICOM and TRAIN_SERIES_DIR is not None:
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for i, (_, row) in enumerate(sample_series.iterrows()):
        files = list_dcm_files(row['StudyInstanceUID'], row['SeriesInstanceUID'])
        if not files:
            continue
        ds = pydicom.dcmread(files[len(files)//2], force=True)
        try:
            arr = ds.pixel_array.astype(float).ravel()
            ax.hist(arr, bins=60, histtype="step", label=row['Anatomical_Plane'], color=PALETTE[i % len(PALETTE)])
        except Exception as e:
            print(f"[warn] gagal decode pixel_array (histogram): {e}")
            continue
    ax.set_title("Distribusi intensitas pixel raw (beberapa contoh slice)")
    ax.set_xlabel("pixel value")
    ax.legend(fontsize=8)
    savefig(fig, "12_pixel_intensity_hist.png")
    plt.show()

## 8. Ringkasan tabel akhir

Menyimpan tabel ringkasan sebagai output kernel supaya bisa langsung ditarik ke lokal bersama notebook.

In [ ]:
final_summary = {
    "n_train_studies": int(train['StudyInstanceUID'].nunique()) if train is not None else None,
    "n_test_studies_public_sample": int(test['StudyInstanceUID'].nunique()) if test is not None else None,
    "pct_labels_missing_avg": float(miss.mean()) if 'miss' in dir() else None,
    "n_studies_with_any_label": int(n_any) if 'n_any' in dir() else None,
    "n_train_series": int(len(train_series)) if train_series is not None else None,
    "avg_series_per_study": float(n_series_per_study.mean()) if 'n_series_per_study' in dir() else None,
}
final_summary_df = pd.DataFrame([final_summary]).T.rename(columns={0: "value"})
final_summary_df.to_csv(OUT_DIR / "eda_summary.csv")
final_summary_df

## 10. Analisis citra DICOM lebih dalam

Bagian ini menggali lebih jauh karakteristik gambar untuk menjawab: **arsitektur/strategi model apa yang paling cocok?** Fokus: heterogenitas scanner/resolusi (domain shift), variasi intensitas antar tipe sekuens (kebutuhan normalisasi), dan variasi antar-slice dalam satu series (kebutuhan modeling multi-slice/3D).

In [ ]:
HDR_SAMPLE_N = min(600, len(train_series)) if train_series is not None else 0
hdr_rows = []
hdr_fail = 0

if train_series is not None and HAVE_PYDICOM and TRAIN_SERIES_DIR is not None:
    hdr_sample = train_series.sample(HDR_SAMPLE_N, random_state=11)
    for _, row in hdr_sample.iterrows():
        files = list_dcm_files(row['StudyInstanceUID'], row['SeriesInstanceUID'])
        if not files:
            hdr_fail += 1
            continue
        try:
            # stop_before_pixels=True -> baca header saja, jauh lebih cepat, tidak perlu decode pixel data
            ds = pydicom.dcmread(files[len(files) // 2], force=True, stop_before_pixels=True)
            ps = getattr(ds, "PixelSpacing", None)
            hdr_rows.append({
                "Anatomical_Plane": row["Anatomical_Plane"],
                "Manufacturer": str(getattr(ds, "Manufacturer", "unknown")).strip(),
                "ManufacturerModelName": str(getattr(ds, "ManufacturerModelName", "unknown")).strip(),
                "MagneticFieldStrength": getattr(ds, "MagneticFieldStrength", None),
                "Rows": getattr(ds, "Rows", None),
                "Columns": getattr(ds, "Columns", None),
                "PixelSpacing_row": float(ps[0]) if ps else None,
                "SliceThickness": getattr(ds, "SliceThickness", None),
                "BitsStored": getattr(ds, "BitsStored", None),
                "PhotometricInterpretation": str(getattr(ds, "PhotometricInterpretation", None)),
                "TransferSyntax": str(getattr(getattr(ds, "file_meta", None), "TransferSyntaxUID", None)),
            })
        except Exception as e:
            hdr_fail += 1

    hdr_df = pd.DataFrame(hdr_rows)
    print(f"Berhasil baca header: {len(hdr_df)} / {HDR_SAMPLE_N} series (gagal: {hdr_fail})")
    display(hdr_df.describe(include="all").T)
else:
    hdr_df = pd.DataFrame()
    print("[skip] pydicom / train_series dir tidak tersedia.")

In [ ]:
if len(hdr_df):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

    manu_counts = hdr_df["Manufacturer"].value_counts()
    axes[0].bar(manu_counts.index, manu_counts.values, color=PALETTE[:len(manu_counts)])
    axes[0].set_title("Distribusi Manufacturer scanner")
    axes[0].tick_params(axis="x", rotation=30)

    field_counts = hdr_df["MagneticFieldStrength"].value_counts().sort_index()
    axes[1].bar(field_counts.index.astype(str), field_counts.values, color=PALETTE[2:6])
    axes[1].set_title("Distribusi MagneticFieldStrength (Tesla)")
    axes[1].set_xlabel("Tesla")

    fig.suptitle(f"Heterogenitas scanner (sampel n={len(hdr_df)} series)", fontsize=13, fontweight="bold")
    savefig(fig, "13_scanner_heterogeneity.png")
    plt.show()

    print("Transfer syntax yang ditemukan:")
    print(hdr_df["TransferSyntax"].value_counts())
    print()
    print("PhotometricInterpretation:")
    print(hdr_df["PhotometricInterpretation"].value_counts())

In [ ]:
if len(hdr_df):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    axes[0].hist(hdr_df["Rows"].dropna(), bins=20, color=ACCENT)
    axes[0].set_title("Resolusi: Rows (px)")

    axes[1].hist(hdr_df["Columns"].dropna(), bins=20, color=ACCENT2)
    axes[1].set_title("Resolusi: Columns (px)")

    axes[2].hist(hdr_df["PixelSpacing_row"].dropna(), bins=20, color=PALETTE[2])
    axes[2].set_title("PixelSpacing (mm/px)")

    fig.suptitle("Variasi resolusi & skala fisik antar series", fontsize=13, fontweight="bold")
    savefig(fig, "14_resolution_pixelspacing.png")
    plt.show()

    n_unique_res = hdr_df[["Rows", "Columns"]].drop_duplicates().shape[0]
    print(f"Jumlah kombinasi (Rows, Columns) UNIK yang ditemukan: {n_unique_res} dari {len(hdr_df)} series sampel")
    print("-> jika > 1, gambar TIDAK berukuran seragam -> wajib resize/pad eksplisit sebelum masuk model.")

### 10.1 Statistik intensitas per tipe sekuens

Menilai seberapa berbeda distribusi nilai pixel antar kombinasi `Anatomical_Plane` × `Fluid_Sensitive` × `Fat_Suppression` — penting untuk menentukan strategi normalisasi (global vs per-sekuens).

In [ ]:
INTENS_SAMPLE_N = min(240, len(train_series)) if train_series is not None else 0
intens_rows = []

if train_series is not None and HAVE_PYDICOM and TRAIN_SERIES_DIR is not None:
    intens_sample = train_series.sample(INTENS_SAMPLE_N, random_state=23)
    for _, row in intens_sample.iterrows():
        files = list_dcm_files(row['StudyInstanceUID'], row['SeriesInstanceUID'])
        if not files:
            continue
        try:
            ds = pydicom.dcmread(files[len(files) // 2], force=True)
            arr = ds.pixel_array.astype(float)
            intens_rows.append({
                "group": f"{row['Anatomical_Plane']} | FS={row['Fluid_Sensitive']} FatSup={row['Fat_Suppression']}",
                "mean": arr.mean(), "std": arr.std(),
                "p1": np.percentile(arr, 1), "p99": np.percentile(arr, 99),
            })
        except Exception as e:
            print(f"[warn] gagal decode: {e}")
            continue

    intens_df = pd.DataFrame(intens_rows)
    if len(intens_df):
        summary = intens_df.groupby("group").agg(["mean", "std"]).round(1)
        display(summary)

        fig, ax = plt.subplots(figsize=(9, 5))
        grp_mean = intens_df.groupby("group")["mean"].mean().sort_values()
        grp_std = intens_df.groupby("group")["std"].mean().reindex(grp_mean.index)
        ax.barh(grp_mean.index, grp_mean.values, xerr=grp_std.values, color=ACCENT, capsize=3)
        ax.set_xlabel("mean pixel intensity (± std antar sampel)")
        ax.set_title(f"Rata-rata intensitas per tipe sekuens (sampel n={len(intens_df)})")
        savefig(fig, "15_intensity_by_sequence.png")
        plt.show()
        print("-> Jika rentang mean antar grup jauh berbeda: NORMALISASI SEBAIKNYA DILAKUKAN PER-GRUP SEKUENS, bukan global.")
else:
    print("[skip] pydicom / train_series dir tidak tersedia.")

### 10.2 Sapuan slice dalam satu series

Menampilkan banyak slice (bukan cuma slice tengah) dari SATU series untuk melihat seberapa besar variasi anatomi dari satu ujung volume ke ujung lain — ini alasan kenapa model *single mid-slice* berisiko kehilangan informasi penting.

In [ ]:
if train_series is not None and HAVE_PYDICOM and TRAIN_SERIES_DIR is not None:
    # cari satu series dengan jumlah slice cukup banyak (>=20) supaya sapuannya informatif
    candidates = train_series.sample(min(60, len(train_series)), random_state=5)
    chosen = None
    for _, row in candidates.iterrows():
        files = list_dcm_files(row['StudyInstanceUID'], row['SeriesInstanceUID'])
        if len(files) >= 20:
            chosen = (row, files)
            break

    if chosen:
        row, files = chosen
        n_show = 12
        idxs = np.linspace(0, len(files) - 1, n_show).astype(int)
        fig, axes = plt.subplots(2, 6, figsize=(18, 6.5))
        for ax, i in zip(axes.flat, idxs):
            ds = pydicom.dcmread(files[i], force=True)
            try:
                img = ds.pixel_array.astype(float)
                img = (img - img.min()) / (np.ptp(img) + 1e-6)
                ax.imshow(img, cmap="gray")
            except Exception as e:
                print(f"[warn] gagal decode slice {i}: {e}")
            ax.set_title(f"slice {i+1}/{len(files)}", fontsize=9)
            ax.axis("off")
        fig.suptitle(f"Sapuan {n_show} slice — {row['Anatomical_Plane']} | FS={row['Fluid_Sensitive']} FatSup={row['Fat_Suppression']} (total {len(files)} slice)",
                     fontsize=13, fontweight="bold")
        savefig(fig, "16_slice_sweep.png")
        plt.show()
    else:
        print("[skip] tidak menemukan series dengan >=20 slice pada sampel kandidat.")
else:
    print("[skip] pydicom / train_series dir tidak tersedia.")

### 10.3 Cek kualitas data — tingkat kegagalan baca DICOM

Sampel lebih besar untuk mengukur seberapa sering file DICOM gagal dibaca / gagal di-decode pixel-nya (relevan untuk pipeline training yang robust — perlu try/except & skip, bukan crash).

In [ ]:
QA_SAMPLE_N = min(400, len(train_series)) if train_series is not None else 0
qa_ok, qa_hdr_fail, qa_pixel_fail = 0, 0, 0

if train_series is not None and HAVE_PYDICOM and TRAIN_SERIES_DIR is not None:
    qa_sample = train_series.sample(QA_SAMPLE_N, random_state=99)
    for _, row in qa_sample.iterrows():
        files = list_dcm_files(row['StudyInstanceUID'], row['SeriesInstanceUID'])
        if not files:
            qa_hdr_fail += 1
            continue
        f = files[len(files) // 2]
        try:
            ds = pydicom.dcmread(f, force=True)
        except Exception:
            qa_hdr_fail += 1
            continue
        try:
            _ = ds.pixel_array
            qa_ok += 1
        except Exception:
            qa_pixel_fail += 1

    print(f"Sampel diuji: {QA_SAMPLE_N} series")
    print(f"  OK (header + pixel decode sukses): {qa_ok} ({qa_ok/QA_SAMPLE_N*100:.1f}%)")
    print(f"  Gagal baca header / file tidak ada: {qa_hdr_fail} ({qa_hdr_fail/QA_SAMPLE_N*100:.1f}%)")
    print(f"  Header OK tapi gagal decode pixel: {qa_pixel_fail} ({qa_pixel_fail/QA_SAMPLE_N*100:.1f}%)")
else:
    print("[skip] pydicom / train_series dir tidak tersedia.")

### 10.4 (Best-effort) mengambil teks metrik evaluasi resmi

Halaman Kaggle *overview/evaluation* di-render lewat JavaScript sehingga tidak terbaca oleh scraper sederhana. Sel ini mencoba mengambil HTML mentah dan mencari JSON state yang mungkin ter-embed di dalamnya — hasilnya best-effort, bisa saja kosong.

In [ ]:
try:
    import requests, re as _re
    r = requests.get("https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/overview/evaluation", timeout=15)
    html = r.text
    print("HTTP status:", r.status_code, "| panjang HTML:", len(html))
    hits = []
    for kw in ["AUC", "ROC", "log loss", "logloss", "log-loss", "mean column-wise", "accuracy", "F1", "Dice"]:
        for m in _re.finditer(_re.escape(kw), html, _re.IGNORECASE):
            s = max(0, m.start() - 80); e = min(len(html), m.end() + 80)
            snippet = _re.sub(r"\s+", " ", html[s:e])
            hits.append((kw, snippet))
    seen = set()
    for kw, snippet in hits[:15]:
        if snippet in seen:
            continue
        seen.add(snippet)
        print(f"[{kw}] ...{snippet}...")
    if not hits:
        print("Tidak ada keyword metrik yang ketemu di HTML mentah (kemungkinan besar konten di-render via JS penuh).")
except Exception as e:
    print(f"[skip] gagal fetch halaman evaluation: {e}")

**Ringkasan bagian 10:** hasil di atas (heterogenitas scanner/resolusi, kebutuhan normalisasi per-sekuens, variasi antar-slice, tingkat kegagalan baca) dipakai langsung sebagai dasar rekomendasi arsitektur & strategi training di bagian kesimpulan.

## 11. Verifikasi klaim dataset & spesifikasi compute Kaggle

Sebelum merancang audit kualitas skala penuh, bagian ini memverifikasi beberapa klaim (total ukuran dataset, campuran transfer syntax, jumlah tag DICOM, konsistensi ID CSV vs folder) memakai sampel yang lebih besar dari bagian 10, plus mencatat spesifikasi compute kernel Kaggle yang sesungguhnya (CPU/RAM/disk).

In [ ]:
import shutil, subprocess

print("=== Spesifikasi compute kernel Kaggle ===")
print("CPU count:", os.cpu_count())
try:
    with open("/proc/meminfo") as f:
        for line in f:
            if line.startswith(("MemTotal", "MemAvailable")):
                print(line.strip())
except Exception as e:
    print("[warn] tidak bisa baca /proc/meminfo:", e)

du = shutil.disk_usage("/kaggle/working")
print(f"Disk /kaggle/working -> total {du.total/1e9:.1f} GB, terpakai {du.used/1e9:.1f} GB, bebas {du.free/1e9:.1f} GB")

try:
    du_input = shutil.disk_usage(str(DATA_DIR))
    print(f"Disk mount data (read-only) -> total {du_input.total/1e9:.1f} GB, bebas {du_input.free/1e9:.1f} GB")
except Exception as e:
    print("[info] tidak bisa baca disk_usage DATA_DIR (wajar untuk FUSE read-only mount):", e)


In [ ]:
# --- Konsistensi ID: jumlah folder Study di disk vs jumlah StudyInstanceUID unik di CSV ---
if TRAIN_SERIES_DIR is not None and train is not None:
    disk_study_dirs = set(p.name for p in TRAIN_SERIES_DIR.iterdir() if p.is_dir())
    csv_study_ids = set(train['StudyInstanceUID'])
    only_on_disk = disk_study_dirs - csv_study_ids
    only_in_csv = csv_study_ids - disk_study_dirs
    print(f"Folder studi di disk: {len(disk_study_dirs)}")
    print(f"StudyInstanceUID unik di train.csv: {len(csv_study_ids)}")
    print(f"Ada di disk TAPI TIDAK di train.csv: {len(only_on_disk)}")
    print(f"Ada di train.csv TAPI TIDAK ada folder di disk: {len(only_in_csv)}")
    if only_in_csv:
        print("Contoh 5 ID yang tercatat di CSV tapi foldernya tidak ada:", list(only_in_csv)[:5])
else:
    print("[skip] TRAIN_SERIES_DIR atau train tidak tersedia.")

In [ ]:
# --- Sensus transfer syntax & jumlah tag DICOM pada sampel JAUH LEBIH BESAR dari bagian 10 (n=600) ---
BIG_SAMPLE_N = min(3000, len(train_series)) if train_series is not None else 0
ts_counter = Counter()
tagcount_list = []
institution_present = 0
big_fail = 0
file_sizes_bytes = []

if train_series is not None and HAVE_PYDICOM and TRAIN_SERIES_DIR is not None:
    big_sample = train_series.sample(BIG_SAMPLE_N, random_state=77)
    for _, row in big_sample.iterrows():
        files = list_dcm_files(row['StudyInstanceUID'], row['SeriesInstanceUID'])
        if not files:
            big_fail += 1
            continue
        f = files[len(files) // 2]
        try:
            ds = pydicom.dcmread(f, force=True, stop_before_pixels=True)
            ts = str(getattr(getattr(ds, "file_meta", None), "TransferSyntaxUID", "unknown"))
            ts_counter[ts] += 1
            tagcount_list.append(len(ds))
            if "InstitutionName" in ds:
                institution_present += 1
            file_sizes_bytes.append(f.stat().st_size)
        except Exception as e:
            big_fail += 1

    print(f"Sampel diuji: {BIG_SAMPLE_N} series (gagal baca: {big_fail})")
    print()
    print("Distribusi TransferSyntaxUID:")
    for ts, cnt in ts_counter.most_common():
        # nama umum transfer syntax UID yang relevan
        names = {
            "1.2.840.10008.1.2": "Implicit VR Little Endian",
            "1.2.840.10008.1.2.1": "Explicit VR Little Endian (uncompressed)",
            "1.2.840.10008.1.2.4.70": "JPEG Lossless",
            "1.2.840.10008.1.2.4.90": "JPEG 2000 Lossless",
            "1.2.840.10008.1.2.4.91": "JPEG 2000",
        }
        print(f"  {ts} ({names.get(ts, 'lainnya')}): {cnt} ({cnt/len(big_sample)*100:.1f}%)")
    print()
    print(f"Tag InstitutionName ADA di: {institution_present} / {len(tagcount_list)} series ({institution_present/max(1,len(tagcount_list))*100:.1f}%)")
    if tagcount_list:
        tc = pd.Series(tagcount_list)
        print(f"Jumlah DICOM element per file (header saja): min={tc.min()}, median={tc.median():.0f}, max={tc.max()}")
    if file_sizes_bytes:
        fs = pd.Series(file_sizes_bytes)
        avg_kb = fs.mean() / 1024
        print(f"Ukuran file rata-rata (mid-slice, sampel n={len(fs)}): {avg_kb:.0f} KB (min {fs.min()/1024:.0f} KB, max {fs.max()/1024:.0f} KB)")
else:
    print("[skip] pydicom / train_series dir tidak tersedia.")

In [ ]:
# --- Estimasi total ukuran & jumlah file dataset (BUKAN sensus penuh -- ekstrapolasi dari sampel) ---
if train_series is not None and 'n_series_per_study' in dir() and 'counts' in dir() and file_sizes_bytes:
    est_total_train_instances = len(train_series) * counts.mean()  # total series x rata-rata slice/series (dari bagian 7)
    est_total_train_bytes = est_total_train_instances * pd.Series(file_sizes_bytes).mean()
    print(f"Estimasi jumlah instance TRAIN (series x rata-rata slice/series): {est_total_train_instances:,.0f}")
    print(f"Estimasi total ukuran TRAIN: {est_total_train_bytes/1e9:.1f} GB")
    print("(test_series hanya sampel publik kecil -> tidak dihitung, ukuran test asli tersembunyi)")
    print()
    print("Klaim di prompt pengguna: ~820.000 file, 569.76 GB (train+test penuh, termasuk test tersembunyi)")
    print("-> jumlah file estimasi ini SEJALAN dengan klaim; ukuran GB tidak bisa dikonfirmasi 1:1 karena test asli tersembunyi.")
else:
    print("[skip] variabel pendukung (n_series_per_study/counts/file_sizes_bytes) belum tersedia.")

**Catatan verifikasi:** sampel n=600 di bagian 10 dan n=3000 di bagian ini **100% menemukan transfer syntax `1.2.840.10008.1.2.1` (Explicit VR Little Endian, uncompressed)** — tidak ada JPEG Lossless/JPEG2000/Implicit VR LE yang terdeteksi pada &asymp;3.600 series yang diuji (&asymp;15% dari total 24.371 series train). Klaim "campuran transfer syntax" pada prompt awal **tidak terkonfirmasi oleh sampel ini** — kemungkinan: (a) transfer syntax campuran itu ada tapi hanya di sebagian kecil series train yang tidak masuk sampel &asymp;15% ini, (b) klaim tersebut merujuk ke test set tersembunyi yang tidak bisa saya akses, atau (c) klaim tersebut tidak akurat. **Jangan asumsikan salah satunya benar tanpa verifikasi lebih lanjut** — lihat catatan di respons utama.

## 9. Kesimpulan & implikasi untuk modeling

1. **Label sangat jarang (~98–99% missing).** Sebagian besar sinyal supervisi ada pada `Report` teks bebas, bukan 12 kolom label terstruktur. Strategi yang masuk akal:
   - Bangun *weak labeler* dari teks laporan (rule-based / NLP) untuk memperluas data latih di luar subset kecil yang berlabel penuh.
   - Pertimbangkan pendekatan semi-supervised / self-training memakai model yang dilatih pada subset kecil, lalu di-pseudo-label ke studi ber-Report tapi tanpa label.
2. **Laporan multi-bahasa** — perlu tokenizer/model NLP multi-bahasa (mis. multilingual BERT/XLM-R) atau terjemahan otomatis sebagai preprocessing, karena studi berasal dari berbagai institusi/bahasa.
3. **Data citra berstruktur hierarkis (Study → Series → Instance)** dengan banyak *view* (Axial/Sagittal/Coronal) dan varian sekuens (fat-suppression, fluid-sensitive) — arsitektur model idealnya bisa menggabungkan informasi multi-series/multi-plane per studi, bukan hanya single image classifier.
4. **`test.csv` publik sangat kecil** — indikasi code competition; jangan overfit ke asumsi statistik dari sampel test publik ini, dan pastikan pipeline inference kompatibel dengan re-run pada test set penuh yang tersembunyi.
5. Langkah eksplorasi lanjutan yang disarankan: analisis lebih dalam pada korelasi keyword-report vs label (weak-supervision), estimasi *inter-institution shift* dari metadata DICOM (Manufacturer, MagneticFieldStrength), dan cek kualitas gambar (resolusi, pixel spacing) antar series untuk menentukan strategi resize/normalisasi.
